<a href="https://colab.research.google.com/github/bsyeol/26AI_project/blob/main/notebooks/2024Q1_all_fpr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06. 전체 데이터 실험: B0 vs E4 (FPR 평가)

## 이 노트북이 하는 일
전체 거래로 B0와 E4를 다시 실행해 거래별 예측값을 저장하고, 평가 기준을 다음과 같이 바꿉니다.

| 구분 | 지표 | 뜻 |
|---|---|---|
| 주지표 | PR-AUC | 모든 threshold를 종합한 성능 |
| 보조 지표 | **Recall@FPR 0.5%**, Recall@FPR 0.1% | 정상 거래를 같은 비율만큼 잘못 찍을 때 이상거래를 얼마나 잡는가 |
| 운영 참고값 | val threshold의 test 결과 | 실제 운영처럼 미리 정한 threshold를 적용하면 어떻게 되는가 |

## 미리 정한 판단 규칙 (결과를 보기 전에 고정)
- **우수 판단:** 같은 FPR에서 Recall이 높은 모델이 우수합니다. 다만 차이의 95% 구간 하한이 0보다 커야 "앞선다"고 말합니다.
- **결론의 우선순위:** PR-AUC가 주지표입니다. Recall@FPR 0.5%가 PR-AUC와 다르게 나오면 "그 오탐률 구간에서만"이라고 범위를 밝힙니다.
- **FPR 값:** 0.1%, 0.5%. 전체 정상 거래 중 200만 원 이상이 약 0.9%이므로 그보다 작은 값을 씁니다.
- **threshold:** Recall@FPR은 각 평가 구간의 정상 거래로 FPR을 정확히 맞춰 모델을 비교합니다. 이 값은 threshold를 고르는 것이 아니라 성능 곡선의 한 점을 읽는 것입니다. 운영 참고값은 val에서 정한 threshold를 test에 적용합니다.

## 실행 전 준비
- 런타임 유형: GPU
- Colab 보안 비밀 `TABPFN_TOKEN` 저장 + 이 노트북의 노트북 액세스 켜기
- 드라이브 `processed/mid/`에 데이터 정리 노트북의 결과 파일과 `enc_E4_seed*.pt`가 있어야 합니다 (인코더가 없으면 새로 학습합니다)

## 저장되는 파일 (`processed/mid/`, 기존 결과는 덮어쓰지 않음)
| 파일 | 내용 |
|---|---|
| `full_pred_{모델}_{val 또는 test}_seed*.csv` | 거래별 예측 확률 |
| `full_results_prauc.csv`, `full_results_fpr.csv`, `full_results_boot.csv` | 항목별 결과 |
| `full_summary_B0_E4.csv` | 모든 결과를 합친 파일 (발표 자료 작성용) |

---
# 공통 준비

In [4]:
# =====================================================================
# [공통-1] 패키지 설치  ── 공통 (B0, E4 모두 사용)
# - B0과 같은 TabPFN 버전을 설치합니다. 버전이 다르면 공정한 비교가 아닙니다.
# - B0, E4 노트북과 같은 버전이어야 결과를 비교할 수 있습니다.
# - 런타임 유형이 GPU인지 먼저 확인하세요.
# =====================================================================
!pip install -q tabpfn==9.2.0

In [5]:
# =====================================================================
# [공통-2] 공통 데이터 불러오기 (B0, E4 노트북과 동일)  ── 공통 (B0, E4 모두 사용)
# - 데이터 정리 노트북이 저장한 파일을 읽고 MD5로 같은 파일인지 확인합니다.
# - settings.json 값(SEEDS, K, H_DIM, FLIP_R, T2V_K, EPOCHS 등)을 변수로 만듭니다.
# - 원본에 없는 feature 두 개(log_amount, dow)를 만듭니다.
# =====================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, time, copy
import numpy as np
import pandas as pd

MID_DIR = '/content/drive/MyDrive/26-2_AI_project/processed/mid'
info = json.load(open(f'{MID_DIR}/split_info.json', encoding='utf-8'))
S    = json.load(open(f'{MID_DIR}/settings.json', encoding='utf-8'))
globals().update(S)

csv_path = f'{MID_DIR}/mid_v1.csv'
assert hashlib.md5(open(csv_path, 'rb').read()).hexdigest() == info['md5'], '파일이 다릅니다'

mid = pd.read_csv(csv_path, encoding='utf-8-sig',
                  dtype={USER: str, '입금계좌일련번호': str}, parse_dates=['date'])
mid['log_amount'] = np.log1p(mid['거래금액'])
mid['dow'] = mid['date'].dt.dayofweek

CONTEXT = {s: np.load(f'{MID_DIR}/context_seed{s}.npy') for s in SEEDS}

print(mid['split'].value_counts().to_dict(), '| 기대값:', info['rows'])
print('context:', {s: len(v) for s, v in CONTEXT.items()})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'train': 229653, 'test': 50264, 'val': 49404} | 기대값: {'train': 229653, 'test': 50264, 'val': 49404}
context: {0: 9996, 1: 9996, 2: 9996}


In [6]:
# =====================================================================
# [공통-3] TabPFN, PyTorch 준비  ── 공통 (B0, E4 모두 사용)
# - 보안 비밀 TABPFN_TOKEN을 읽어 라이선스 인증을 합니다.
# - GPU를 확인합니다. 인코더 학습도 GPU에서 해야 시간이 맞습니다.
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

from google.colab import userdata
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN').strip()
from tabpfn import TabPFNClassifier
import tabpfn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| tabpfn', tabpfn.__version__)

device: cuda | tabpfn 9.2.0


In [7]:
# =====================================================================
# [공통-4] 금액 규칙 기준선과 TabPFN 실행 함수  ── 공통 (B0, E4 모두 사용)
# - BIG: train 이상거래의 최소 금액. val, test 라벨은 쓰지 않습니다.
# - 금액 규칙: 거래금액이 BIG 이상이면 이상거래로 찍는 규칙입니다.
# - run_full: seed별 고정 context로 TabPFN을 실행하고 val, test 예측 확률을 저장합니다.
#   B0, E4 노트북과 같은 context를 쓰므로 PR-AUC가 기존 결과와 비슷해야 합니다.
# =====================================================================
is_fraud = (mid[LABEL] == 1).to_numpy()
is_train = (mid['split'] == 'train').to_numpy()
BIG = int(mid.loc[is_train & is_fraud, '거래금액'].min())
print(f'train 이상거래 최소 금액: {BIG:,}원')

rule_rows = {}
for part in ['val', 'test']:
    d = mid[mid['split'] == part]
    pred, y = d['거래금액'] >= BIG, d[LABEL] == 1
    tp, fp = int((pred & y).sum()), int((pred & ~y).sum())
    prec, rec = tp / max(tp + fp, 1), tp / y.sum()
    rule_rows[part] = {'precision': prec, 'recall': rec, 'f1': 2 * prec * rec / (prec + rec), 'fpr': fp / (~y).sum()}
print('금액 규칙:', pd.DataFrame(rule_rows).round(4).T.to_string())

def pred_path(model, part, seed):
    return f'{MID_DIR}/full_pred_{model}_{part}_seed{seed}.csv'

def run_full(feats, seed, model):
    ctx = CONTEXT[seed]
    clf = TabPFNClassifier(random_state=seed)
    clf.fit(mid.loc[ctx, feats].to_numpy(np.float32), mid.loc[ctx, LABEL].to_numpy())
    r = {'model': model, 'seed': seed}
    for part in ['val', 'test']:
        idx = np.where((mid['split'] == part).to_numpy())[0]
        X = mid.loc[idx, feats].to_numpy(np.float32)
        p = np.concatenate([clf.predict_proba(X[i:i + PRED_BATCH])[:, 1] for i in range(0, len(X), PRED_BATCH)])
        y = mid.loc[idx, LABEL].to_numpy()
        pd.DataFrame({'row': idx, 'proba': p, 'label': y}).to_csv(pred_path(model, part, seed), index=False)
        r[f'{part}_pr_auc'], r[f'{part}_roc_auc'] = average_precision_score(y, p), roc_auc_score(y, p)
    return r

results = []

train 이상거래 최소 금액: 2,000,000원
금액 규칙:       precision  recall      f1     fpr
val      0.4673     1.0  0.6370  0.0090
test     0.4197     1.0  0.5912  0.0087


---
# B0: 이력 없음 (베이스)

In [8]:
# =====================================================================
# [B0-1] B0 실행  ── B0 전용 (인코더 없음)
# - 현재 거래 정보 7개(BASE_FEATS)만 사용합니다.
# =====================================================================
for seed in SEEDS:
    t0 = time.time()
    r = run_full(BASE_FEATS, seed, 'B0')
    results.append(r)
    print(f'B0 seed {seed}: val PR-AUC {r["val_pr_auc"]:.4f} | test PR-AUC {r["test_pr_auc"]:.4f}  ({time.time() - t0:.0f}s)')

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

B0 seed 0: val PR-AUC 0.5238 | test PR-AUC 0.5735  (186s)
B0 seed 1: val PR-AUC 0.5271 | test PR-AUC 0.6178  (179s)
B0 seed 2: val PR-AUC 0.5569 | test PR-AUC 0.5977  (179s)


---
# E4: 이력 h 추가
`04` 노트북과 같은 코드입니다. 저장된 인코더를 불러오면 같은 h가 나옵니다.

In [9]:
# =====================================================================
# [E4-1] (계좌, 3시간 bin) 집계  ── E4 전용 (B0은 사용 안 함)
# - 같은 계좌, 같은 3시간 구간의 거래를 한 행으로 묶습니다.
#   같은 구간 안의 거래 순서는 알 수 없으므로 순서 없이 요약합니다.
# - cnt: 건수 / amt_sum, amt_mean, amt_max: 금액 합, 평균, 최대
#   n_recv: 고유 수취계좌 수 / n_inst: 고유 입금기관 수
#   gap: 같은 계좌의 직전 거래 구간 이후 지난 구간 수
#   bin_of_day, dow: 시간대(0~7), 요일(0~6) -> Time2Vec 입력
# - 라벨(이상거래여부)은 집계에 쓰지 않습니다.
# =====================================================================
bins = (mid.groupby([USER, 'bin_id'])
          .agg(cnt=('date', 'size'),
               amt_sum=('거래금액', 'sum'), amt_mean=('거래금액', 'mean'), amt_max=('거래금액', 'max'),
               n_recv=('입금계좌일련번호', 'nunique'), n_inst=('입금금융회사일련번호', 'nunique'),
               bin_of_day=('bin_of_day', 'first'), date=('date', 'first'), split=('split', 'first'))
          .reset_index()
          .sort_values([USER, 'bin_id'], kind='stable')
          .reset_index(drop=True))
bins['gap'] = bins.groupby(USER)['bin_id'].diff().fillna(0)
bins['dow'] = bins['date'].dt.dayofweek

print('bin 수:', len(bins), '| 계좌 수:', bins[USER].nunique())
print('계좌당 bin 수:', bins.groupby(USER).size().describe(percentiles=[.25, .5, .75]).round(1).to_dict())

bin 수: 287660 | 계좌 수: 27694
계좌당 bin 수: {'count': 27694.0, 'mean': 10.4, 'std': 10.6, 'min': 1.0, '25%': 3.0, '50%': 8.0, '75%': 15.0, 'max': 204.0}


In [10]:
# =====================================================================
# [E4-2] 정규화  ── E4 전용 (B0은 사용 안 함)
# - 건수, 금액, 간격을 log 변환한 뒤 표준화합니다.
# - 평균과 표준편차는 train 구간에서만 계산합니다(val, test 정보 누수 방지).
# - X: 시퀀스 feature (bin 수 x 7), T: Time2Vec 입력 (bin 수 x 2)
# =====================================================================
SEQ_FEATS = ['cnt', 'amt_sum', 'amt_mean', 'amt_max', 'n_recv', 'n_inst', 'gap']

Z = np.log1p(bins[SEQ_FEATS].to_numpy(np.float64))
is_tr = (bins['split'] == 'train').to_numpy()
mu, sd = Z[is_tr].mean(0), Z[is_tr].std(0) + 1e-6
X = ((Z - mu) / sd).astype(np.float32)
T = bins[['bin_of_day', 'dow']].to_numpy(np.float32)
print('train 기준 평균:', dict(zip(SEQ_FEATS, np.round(mu, 2))))

train 기준 평균: {'cnt': np.float64(0.73), 'amt_sum': np.float64(11.16), 'amt_mean': np.float64(11.11), 'amt_max': np.float64(11.12), 'n_recv': np.float64(0.73), 'n_inst': np.float64(0.72), 'gap': np.float64(2.77)}


In [11]:
# =====================================================================
# [E4-3] 시퀀스 창 만들기  ── E4 전용 (B0은 사용 안 함)
# - 각 (계좌, bin) 행마다 같은 계좌의 바로 이전 K개 bin의 행 번호를 모읍니다.
# - 현재 bin은 넣지 않습니다(strict). 판단 시점에 알 수 있는 정보만 씁니다.
# - 이력이 K개보다 짧으면 앞쪽을 0 벡터로 채웁니다(행 번호 -1).
# - 사전학습 정답은 현재 bin의 집계값입니다("이전 K개로 다음 bin 예측").
# - TR_ROWS: train 구간 학습 샘플, VA_ROWS: val 구간 early stopping용 샘플
# =====================================================================
N = len(bins)
pos = bins.groupby(USER).cumcount().to_numpy()
WIN = np.full((N, K), -1, dtype=np.int64)
for k in range(1, K + 1):
    ok = pos >= k
    WIN[ok, K - k] = np.arange(N)[ok] - k

X_pad = np.vstack([X, np.zeros((1, X.shape[1]), np.float32)])
T_pad = np.vstack([T, np.zeros((1, T.shape[1]), np.float32)])

def batch(rows):
    w = WIN[rows]
    return (torch.from_numpy(X_pad[w]).to(DEVICE),
            torch.from_numpy(T_pad[w]).to(DEVICE),
            torch.from_numpy(X[rows]).to(DEVICE))

has_hist = pos >= 1
TR_ROWS = np.where(has_hist & (bins['split'] == 'train').to_numpy())[0]
VA_ROWS = np.where(has_hist & (bins['split'] == 'val').to_numpy())[0]
print('사전학습 train:', len(TR_ROWS), '| val:', len(VA_ROWS))
print('이력 없는 bin 비율 (계좌의 첫 bin):', f'{(pos == 0).mean():.1%}')
print(f'이력이 K={K}개 이상인 bin 비율:', f'{(pos >= K).mean():.1%}')

사전학습 train: 174973 | val: 42251
이력 없는 bin 비율 (계좌의 첫 bin): 9.6%
이력이 K=16개 이상인 bin 비율: 19.3%


In [12]:
# =====================================================================
# [E4-4] 인코더 정의: Time2Vec + SIGMA  ── E4 전용 (B0은 사용 안 함)
# - Time2Vec: 시간대, 요일 -> 선형 항 1개 + 학습 가능한 주기(sin) 항.
# - Mamba: selective SSM. 입력에 따라 상태 갱신 정도(dt)가 달라집니다. PyTorch만으로 구현.
# - SIGMA 블록 (Liu et al., AAAI 2025)
#   PF-Mamba: 원래 순서 Mamba + Partial Flip(최근 FLIP_R개 유지, 나머지 역순) Mamba
#   DS Gate: 두 Mamba 출력의 가중치를 입력에 따라 결정
#   FE-GRU: Conv1d + GRU로 짧은 구간 패턴
#   Mixing -> Linear -> PFFN -> 잔차 + LayerNorm
# - Time2Vec은 Partial Flip 이전에 더합니다. 순서를 뒤집어도 각 bin의 시각 정보는 유지됩니다.
# - 출력 h: 마지막 위치(가장 최근 bin)를 H_DIM 차원으로 바꾼 벡터
# - 파라미터 수가 PARAM_BUDGET 범위 밖이면 멈춥니다(모델 간 크기 통일).
# =====================================================================
class Time2Vec(nn.Module):
    def __init__(self, in_dim, k):
        super().__init__()
        self.lin = nn.Linear(in_dim, 1)
        self.per = nn.Linear(in_dim, k - 1)
    def forward(self, t):
        return torch.cat([self.lin(t), torch.sin(self.per(t))], -1)


class Mamba(nn.Module):
    def __init__(self, d, n=16, expand=2, k=4):
        super().__init__()
        di = d * expand
        self.di, self.n = di, n
        self.inp  = nn.Linear(d, 2 * di)
        self.conv = nn.Conv1d(di, di, k, groups=di, padding=k - 1)
        self.xp   = nn.Linear(di, 1 + 2 * n)
        self.dt   = nn.Linear(1, di)
        self.A    = nn.Parameter(-torch.rand(di, n))
        self.out  = nn.Linear(di, d)

    def forward(self, x):
        B, L, _ = x.shape
        u, z = self.inp(x).chunk(2, -1)
        u = F.silu(self.conv(u.transpose(1, 2))[..., :L].transpose(1, 2))
        p = self.xp(u)
        dt, Bm, Cm = p[..., :1], p[..., 1:1 + self.n], p[..., 1 + self.n:]
        dt = F.softplus(self.dt(dt))
        h = x.new_zeros(B, self.di, self.n)
        ys = []
        for t in range(L):
            h = torch.exp(dt[:, t, :, None] * self.A) * h + dt[:, t, :, None] * Bm[:, t, None, :] * u[:, t, :, None]
            ys.append((h * Cm[:, t, None, :]).sum(-1))
        return self.out(torch.stack(ys, 1) * F.silu(z))


class SigmaEncoder(nn.Module):
    def __init__(self, f_in, t_in, d=H_DIM, r=FLIP_R, t2v_k=T2V_K):
        super().__init__()
        self.r = r
        self.emb   = nn.Linear(f_in, d)
        self.t2v   = Time2Vec(t_in, t2v_k)
        self.t_emb = nn.Linear(t2v_k, d)
        self.m_fwd, self.m_flip = Mamba(d), Mamba(d)
        self.g_conv = nn.Conv1d(d, d, 3, padding=1)
        self.g_lin  = nn.Linear(d, d)
        self.f_conv = nn.Conv1d(d, d, 3, padding=1)
        self.gru    = nn.GRU(d, d, batch_first=True)
        self.a      = nn.Parameter(torch.ones(2))
        self.lin    = nn.Linear(d, d)
        self.ffn    = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.norm   = nn.LayerNorm(d)
        self.proj   = nn.Linear(d, H_DIM)

    def gate(self, h):
        g = self.g_lin(self.g_conv(h.transpose(1, 2)).transpose(1, 2))
        return F.silu(g) + torch.sigmoid(g)

    def forward(self, x, t):
        h = self.emb(x) + self.t_emb(self.t2v(t))
        n = h.shape[1] - self.r
        h_flip = torch.cat([h[:, :n].flip(1), h[:, n:]], 1)
        M = self.gate(h) * self.m_fwd(h) + self.gate(h_flip) * self.m_flip(h_flip)
        Fg = self.gru(self.f_conv(h.transpose(1, 2)).transpose(1, 2))[0]
        z = self.lin(self.a[0] * M + self.a[1] * Fg)
        out = self.norm(h + self.ffn(z))
        return self.proj(out[:, -1])


n_params = sum(p.numel() for p in SigmaEncoder(len(SEQ_FEATS), 2).parameters())
print('인코더 파라미터 수:', n_params)
assert PARAM_BUDGET[0] <= n_params <= PARAM_BUDGET[1], '파라미터 수가 허용 범위 밖입니다'

인코더 파라미터 수: 44540


In [13]:
# =====================================================================
# [E4-5] 인코더 사전학습과 h 추출 함수  ── E4 전용 (B0은 사용 안 함)
# - train_encoder: 이전 K개 bin으로 현재 bin 집계값을 예측하도록 학습(Huber loss).
#   이상거래 라벨은 쓰지 않습니다. train 구간 bin으로만 학습합니다.
#   val loss가 PATIENCE번 연속 나아지지 않으면 멈추고 가장 좋았던 가중치로 되돌립니다.
#   학습 후 예측 head는 버리고 인코더만 반환합니다.
# - extract_h: 모든 (계좌, bin)에 대해 h를 계산합니다.
# =====================================================================
def train_encoder(seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    enc  = SigmaEncoder(len(SEQ_FEATS), 2).to(DEVICE)
    head = nn.Linear(H_DIM, len(SEQ_FEATS)).to(DEVICE)
    opt  = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=LR)

    def val_loss():
        enc.eval(); head.eval(); tot = 0.0
        with torch.no_grad():
            for i in range(0, len(VA_ROWS), BATCH):
                xs, ts, y = batch(VA_ROWS[i:i + BATCH])
                tot += F.huber_loss(head(enc(xs, ts)), y, reduction='sum').item()
        return tot / (len(VA_ROWS) * len(SEQ_FEATS))

    best, best_state, wait = float('inf'), None, 0
    for ep in range(EPOCHS):
        enc.train(); head.train(); t0 = time.time()
        perm = rng.permutation(TR_ROWS)
        for i in range(0, len(perm), BATCH):
            xs, ts, y = batch(perm[i:i + BATCH])
            loss = F.huber_loss(head(enc(xs, ts)), y)
            opt.zero_grad(); loss.backward(); opt.step()
        vl = val_loss()
        print(f'  epoch {ep + 1}: val loss {vl:.4f} ({time.time() - t0:.0f}s)')
        if vl < best:
            best, best_state, wait = vl, copy.deepcopy(enc.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    enc.load_state_dict(best_state)
    return enc


def extract_h(enc):
    enc.eval(); out = []
    with torch.no_grad():
        for i in range(0, N, 4096):
            xs, ts, _ = batch(np.arange(i, min(i + 4096, N)))
            out.append(enc(xs, ts).cpu().numpy())
    return np.concatenate(out)

In [15]:
# =====================================================================
# [E4-6] E4 실행  ── E4 전용 (인코더로 만든 h 추가)
# - 저장된 인코더(enc_E4_seed*.pt)가 있으면 불러오고, 없으면 새로 학습해 저장합니다.
# - h를 거래에 붙인 뒤 BASE_FEATS + h로 TabPFN을 실행합니다.
# =====================================================================
H_COLS = [f'h{i}' for i in range(H_DIM)]

for seed in SEEDS:
    t0 = time.time()
    path = f'{MID_DIR}/enc_E4_seed{seed}.pt'
    if os.path.exists(path):
        enc = SigmaEncoder(len(SEQ_FEATS), 2).to(DEVICE)
        enc.load_state_dict(torch.load(path, map_location=DEVICE))
        print(f'E4 seed {seed}: 저장된 인코더 사용')
    else:
        print(f'E4 seed {seed}: 저장된 인코더 없음 -> 새로 학습')
        enc = train_encoder(seed)
        torch.save(enc.state_dict(), path)

    H = pd.DataFrame(extract_h(enc), columns=H_COLS)
    H[USER], H['bin_id'] = bins[USER].values, bins['bin_id'].values
    mid = mid.drop(columns=[c for c in H_COLS if c in mid.columns])
    mid = mid.merge(H, on=[USER, 'bin_id'], how='left', validate='many_to_one')
    assert mid[H_COLS].notna().all().all()

    r = run_full(BASE_FEATS + H_COLS, seed, 'E4')
    results.append(r)
    print(f'E4 seed {seed}: val PR-AUC {r["val_pr_auc"]:.4f} | test PR-AUC {r["test_pr_auc"]:.4f}  ({time.time() - t0:.0f}s)')

E4 seed 0: 저장된 인코더 사용
E4 seed 0: val PR-AUC 0.5712 | test PR-AUC 0.6080  (236s)
E4 seed 1: 저장된 인코더 사용
E4 seed 1: val PR-AUC 0.5886 | test PR-AUC 0.6454  (236s)
E4 seed 2: 저장된 인코더 사용
E4 seed 2: val PR-AUC 0.5734 | test PR-AUC 0.5884  (236s)


---
# 평가

In [16]:
# =====================================================================
# [평가-1] PR-AUC  ── B0, E4 결과 비교
# - 기존 전체 데이터 결과와 비슷한지 먼저 확인합니다.
#   기대값: B0 val 0.536, test 0.596 / E4 val 0.578, test 0.614
# - frame: 한 구간의 라벨, 계좌, 금액, 모델별 seed 평균 확률을 한 표로 만듭니다.
# =====================================================================
res = pd.DataFrame(results)
res.to_csv(f'{MID_DIR}/full_results_prauc.csv', index=False)

def frame(part):
    out = None
    for model in ['B0', 'E4']:
        ps = [pd.read_csv(pred_path(model, part, s)) for s in SEEDS]
        assert all((p['row'] == ps[0]['row']).all() for p in ps)
        if out is None:
            out = ps[0][['row', 'label']].copy()
        out[model] = np.mean([p['proba'].to_numpy() for p in ps], axis=0)
    out[USER] = mid.loc[out['row'], USER].to_numpy()
    out['금액'] = mid.loc[out['row'], '거래금액'].to_numpy()
    return out

DF = {p: frame(p) for p in ['val', 'test']}

rows = []
for part in ['val', 'test']:
    d = DF[part]
    rows.append({'구간': part, '모델': '무작위', 'PR-AUC': d['label'].mean()})
    rows.append({'구간': part, '모델': '금액만', 'PR-AUC': average_precision_score(d['label'], d['금액'])})
    for model in ['B0', 'E4']:
        g = res[res['model'] == model][f'{part}_pr_auc']
        rows.append({'구간': part, '모델': model, 'PR-AUC': g.mean(), '표준편차': g.std()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

  구간  모델  PR-AUC   표준편차
 val 무작위  0.0078    NaN
 val 금액만  0.4821    NaN
 val  B0  0.5359 0.0182
 val  E4  0.5777 0.0095
test 무작위  0.0063    NaN
test 금액만  0.4703    NaN
test  B0  0.5963 0.0222
test  E4  0.6139 0.0290


In [17]:
# =====================================================================
# [평가-2] Recall@FPR  ── B0, E4 결과 비교
# - 비교 표: 평가 구간의 정상 거래로 FPR을 맞춘 뒤 Recall을 비교합니다. 같은 FPR에서 Recall이 높으면 우수합니다.
# - 운영 표: val에서 정한 threshold를 test에 적용합니다. 실제 FPR이 목표와 다를 수 있어 우열 비교에 쓰지 않습니다.
# - F1 표: 기존 방식의 결과입니다. 금액 규칙과 얼마나 비슷한지 확인하는 용도입니다.
# =====================================================================
TARGETS = [0.001, 0.005]

def threshold(score, y, target):
    return np.quantile(score[y == 0], 1 - target, method='higher')

def at_threshold(score, y, t):
    pred = score > t
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    prec, rec = tp / max(tp + fp, 1), tp / (y == 1).sum()
    return {'recall': rec, 'actual_fpr': fp / (y == 0).sum(), 'precision': prec,
            '잡은 이상거래': tp, '잘못 찍은 정상': fp}

def f1_max_threshold(score, y):
    prec, rec, thr = precision_recall_curve(y, score)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return thr[np.argmax(f1[:-1])]

def load(model, part, s):
    p = pd.read_csv(pred_path(model, part, s))
    return p['proba'].to_numpy(), p['label'].to_numpy()

rows = []
for model in ['B0', 'E4']:
    for s in SEEDS:
        pv, yv = load(model, 'val', s)
        pt, yt = load(model, 'test', s)
        for target in TARGETS:
            rows.append({'표': '비교', '모델': model, 'seed': s, 'target_fpr': target, **at_threshold(pt, yt, threshold(pt, yt, target))})
            rows.append({'표': '운영', '모델': model, 'seed': s, 'target_fpr': target, **at_threshold(pt, yt, threshold(pv, yv, target))})
        pred = pt >= f1_max_threshold(pv, yv)
        tp, fp = int((pred & (yt == 1)).sum()), int((pred & (yt == 0)).sum())
        rows.append({'표': 'F1', '모델': model, 'seed': s, 'target_fpr': np.nan, 'recall': tp / (yt == 1).sum(),
                     'actual_fpr': fp / (yt == 0).sum(), 'precision': tp / max(tp + fp, 1), '잡은 이상거래': tp, '잘못 찍은 정상': fp})

av, yv, at, yt = DF['val']['금액'].to_numpy(), DF['val']['label'].to_numpy(), DF['test']['금액'].to_numpy(), DF['test']['label'].to_numpy()
for target in TARGETS:
    rows.append({'표': '비교', '모델': '금액만', 'seed': -1, 'target_fpr': target, **at_threshold(at, yt, threshold(at, yt, target))})
    rows.append({'표': '운영', '모델': '금액만', 'seed': -1, 'target_fpr': target, **at_threshold(at, yt, threshold(av, yv, target))})

res_fpr = pd.DataFrame(rows)
res_fpr.to_csv(f'{MID_DIR}/full_results_fpr.csv', index=False)
cols = ['recall', 'actual_fpr', 'precision', '잡은 이상거래', '잘못 찍은 정상']
print('[비교 표] 같은 FPR에서 Recall이 높으면 우수 (test, seed 평균)')
print(res_fpr[res_fpr['표'] == '비교'].groupby(['target_fpr', '모델'])[cols].mean().round(4))
print('\n[운영 표] val threshold를 test에 적용 (seed 평균)')
print(res_fpr[res_fpr['표'] == '운영'].groupby(['target_fpr', '모델'])[cols].mean().round(4))
print('\n[F1 표] 기존 방식 (seed 평균)')
print(res_fpr[res_fpr['표'] == 'F1'].groupby('모델')[cols].mean().round(4))
print(f"\n금액 규칙 (test): Precision {rule_rows['test']['precision']:.4f} | Recall {rule_rows['test']['recall']:.4f} "
      f"| F1 {rule_rows['test']['f1']:.4f} | FPR {rule_rows['test']['fpr']:.4f}")

[비교 표] 같은 FPR에서 Recall이 높으면 우수 (test, seed 평균)
                recall  actual_fpr  precision   잡은 이상거래  잘못 찍은 정상
target_fpr 모델                                                    
0.001      B0   0.2732      0.0010     0.6353   86.3333   49.0000
           E4   0.3186      0.0010     0.6703  100.6667   49.0000
           금액만  0.0759      0.0005     0.4706   24.0000   27.0000
0.005      B0   0.8238      0.0050     0.5114  260.3333  248.6667
           E4   0.8101      0.0050     0.5068  256.0000  249.0000
           금액만  0.5348      0.0033     0.5090  169.0000  163.0000

[운영 표] val threshold를 test에 적용 (seed 평균)
                recall  actual_fpr  precision   잡은 이상거래  잘못 찍은 정상
target_fpr 모델                                                    
0.001      B0   0.2257      0.0007     0.6670   71.3333   35.3333
           E4   0.1951      0.0006     0.6831   61.6667   27.6667
           금액만  0.0759      0.0005     0.4706   24.0000   27.0000
0.005      B0   0.7363      0.0045     0.5105  232.66

In [18]:
# =====================================================================
# [평가-3] 부트스트랩: 차이가 우연인지 확인  ── B0, E4 결과 비교
# - 같은 계좌의 거래는 서로 독립이 아니므로 계좌 단위로 복원추출합니다.
# - 같은 계좌 묶음으로 모델들을 비교합니다(대응 비교). seed 평균 확률을 씁니다.
# - Recall@FPR은 다시 뽑은 정상 거래로 매번 FPR을 맞춥니다.
# - 95% 구간 하한이 0보다 크면 "앞선다"고 판단합니다.
# =====================================================================
N_BOOT = 1000
MODELS = ['B0', 'E4', '금액']
PAIRS = [('E4', 'B0'), ('E4', '금액'), ('B0', '금액')]

def metrics(y, score):
    out = {'PR-AUC': average_precision_score(y, score)}
    pos, neg = score[y == 1], score[y == 0]
    for target in TARGETS:
        out[f'Recall@FPR {target:.1%}'] = (pos > np.quantile(neg, 1 - target, method='higher')).mean()
    return out

def boot_part(d, part, seed=0):
    rng = np.random.default_rng(seed)
    g = list(d.groupby(USER).indices.values())
    y = d['label'].to_numpy()
    S = {m: d[m].to_numpy() for m in MODELS}
    point = {m: metrics(y, S[m]) for m in MODELS}
    draws = []
    for _ in range(N_BOOT):
        i = np.concatenate([g[j] for j in rng.integers(0, len(g), len(g))])
        if 0 < y[i].sum() < len(i):
            draws.append({m: metrics(y[i], S[m][i]) for m in MODELS})
    rows = []
    for a, b in PAIRS:
        for key in point[a]:
            ds = np.array([x[a][key] - x[b][key] for x in draws])
            rows.append({'구간': part, '지표': key, '비교': f'{a} - {b}', 'A': point[a][key], 'B': point[b][key],
                         '차이': point[a][key] - point[b][key], '하한': np.percentile(ds, 2.5),
                         '상한': np.percentile(ds, 97.5), '앞선 비율': (ds > 0).mean()})
    return rows

t0 = time.time()
boot = pd.DataFrame(boot_part(DF['val'], 'val') + boot_part(DF['test'], 'test'))
boot.to_csv(f'{MID_DIR}/full_results_boot.csv', index=False)
print(boot.round(4).to_string(index=False))
print(f'\n소요 시간: {time.time() - t0:.0f}s')

print('\n[test 판단] 95% 구간 하한이 0보다 크면 앞섬')
for key in boot['지표'].unique():
    for cmp in boot['비교'].unique():
        low = boot[(boot['구간'] == 'test') & (boot['지표'] == key) & (boot['비교'] == cmp)]['하한'].item()
        print(f'  {key} | {cmp}:', '앞섬' if low > 0 else '확인되지 않음')

  구간              지표      비교      A      B      차이      하한     상한  앞선 비율
 val          PR-AUC E4 - B0 0.5928 0.5513  0.0415  0.0005 0.0950  0.975
 val Recall@FPR 0.1% E4 - B0 0.1658 0.1943 -0.0285 -0.0997 0.1299  0.425
 val Recall@FPR 0.5% E4 - B0 0.7228 0.6658  0.0570 -0.0139 0.1514  0.924
 val          PR-AUC E4 - 금액 0.5928 0.4821  0.1107  0.0501 0.1850  1.000
 val Recall@FPR 0.1% E4 - 금액 0.1658 0.0907  0.0751 -0.0094 0.2486  0.959
 val Recall@FPR 0.5% E4 - 금액 0.7228 0.4197  0.3031  0.1546 0.4278  1.000
 val          PR-AUC B0 - 금액 0.5513 0.4821  0.0692  0.0282 0.1150  1.000
 val Recall@FPR 0.1% B0 - 금액 0.1943 0.0907  0.1036  0.0242 0.1898  0.994
 val Recall@FPR 0.5% B0 - 금액 0.6658 0.4197  0.2461  0.0975 0.3994  1.000
test          PR-AUC E4 - B0 0.6369 0.6096  0.0273 -0.0325 0.0895  0.774
test Recall@FPR 0.1% E4 - B0 0.3703 0.2943  0.0759 -0.0770 0.1960  0.838
test Recall@FPR 0.5% E4 - B0 0.8513 0.8259  0.0253 -0.0683 0.0736  0.547
test          PR-AUC E4 - 금액 0.6369 0.4703  0.1666 

---
# 왜 Recall이 99%였는가
기존 F1 최대 threshold가 실제로 어떤 거래를 찍었는지 확인합니다. 발표에서 원인을 설명하는 근거로 씁니다.

In [19]:
# =====================================================================
# [원인-1] 금액 구간별 분포와 F1 threshold가 찍은 거래  ── B0, E4 결과 비교
# - 첫 표: 금액 기준 미만과 이상 구간의 거래 수, 이상거래 수, 이상거래 비율입니다.
# - 둘째 표: F1 threshold로 test에서 찍은 거래가 금액 기준 이상에 얼마나 몰렸는지 봅니다.
#   "기준 이상 중 찍힌 비율"이 1에 가까우면 고액 거래를 사실상 전부 찍은 것입니다.
# - 전부 찍기 F1: 금액 기준 이상 거래를 전부 찍었을 때의 전체 데이터 F1입니다.
# =====================================================================
rows = []
for part in ['val', 'test']:
    d = mid[mid['split'] == part]
    for name, m in [('기준 미만', d['거래금액'] < BIG), ('기준 이상', d['거래금액'] >= BIG)]:
        rows.append({'구간': part, '금액': name, '거래': int(m.sum()), '이상거래': int(d.loc[m, LABEL].sum()),
                     '이상거래 비율': d.loc[m, LABEL].mean(),
                     '정상 거래 중 비율': ((d[LABEL] == 0) & m).sum() / (d[LABEL] == 0).sum()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

rows = []
for model in ['B0', 'E4']:
    for s in SEEDS:
        pv, yv = load(model, 'val', s)
        te = pd.read_csv(pred_path(model, 'test', s))
        pred = te['proba'].to_numpy() >= f1_max_threshold(pv, yv)
        high = mid.loc[te['row'], '거래금액'].to_numpy() >= BIG
        y = te['label'].to_numpy()
        rows.append({'모델': model, 'seed': s, '찍은 거래': int(pred.sum()),
                     '기준 이상 비율': (pred & high).sum() / max(pred.sum(), 1),
                     '기준 이상 중 찍힌 비율': (pred & high).sum() / high.sum(),
                     '기준 미만에서 찍은 거래': int((pred & ~high).sum()),
                     'recall': (pred & (y == 1)).sum() / (y == 1).sum()})
res_why = pd.DataFrame(rows)
print()
print(res_why.round(4).to_string(index=False))
print('\n금액 기준 이상 전체 test 거래:', int((DF['test']['금액'] >= BIG).sum()), '건')
print('기준 이상을 전부 찍을 때의 F1:', {p: round(rule_rows[p]['f1'], 4) for p in rule_rows})

  구간    금액    거래  이상거래  이상거래 비율  정상 거래 중 비율
 val 기준 미만 48578     0   0.0000      0.9910
 val 기준 이상   826   386   0.4673      0.0090
test 기준 미만 49511     0   0.0000      0.9913
test 기준 이상   753   316   0.4197      0.0087

모델  seed  찍은 거래  기준 이상 비율  기준 이상 중 찍힌 비율  기준 미만에서 찍은 거래  recall
B0     0    707       1.0         0.9389              0  0.9842
B0     1    738       1.0         0.9801              0  1.0000
B0     2    733       1.0         0.9734              0  0.9968
E4     0    640       1.0         0.8499              0  0.9272
E4     1    728       1.0         0.9668              0  0.9905
E4     2    566       1.0         0.7517              0  0.8766

금액 기준 이상 전체 test 거래: 753 건
기준 이상을 전부 찍을 때의 F1: {'val': np.float64(0.637), 'test': np.float64(0.5912)}


In [20]:
# =====================================================================
# [저장] 모든 결과를 CSV 하나로 저장  ── 발표 자료 작성용
# - 파일: full_summary_B0_E4.csv (표, 구간, 대상, 지표, FPR, 값, 표준편차, 하한, 상한)
# - 모델 값은 seed 평균과 표준편차, 비교 차이는 seed 평균 확률의 부트스트랩 결과입니다.
# =====================================================================
rows = []
for model in ['B0', 'E4']:
    g = res[res['model'] == model]
    for part in ['val', 'test']:
        for name in ['pr_auc', 'roc_auc']:
            c = g[f'{part}_{name}']
            rows.append({'표': '성능', '구간': part, '대상': model, '지표': name, '값': c.mean(), '표준편차': c.std()})
for part in ['val', 'test']:
    d = DF[part]
    rows.append({'표': '성능', '구간': part, '대상': '무작위', '지표': 'pr_auc', '값': d['label'].mean()})
    rows.append({'표': '성능', '구간': part, '대상': '금액만', '지표': 'pr_auc', '값': average_precision_score(d['label'], d['금액'])})
    for k, v in rule_rows[part].items():
        rows.append({'표': '금액 규칙', '구간': part, '대상': '금액 규칙', '지표': k, '값': v})
for (tb, fpr, model), g in res_fpr.groupby(['표', 'target_fpr', '모델'], dropna=False):
    for k in ['recall', 'actual_fpr', 'precision', '잡은 이상거래', '잘못 찍은 정상']:
        rows.append({'표': tb, '구간': 'test', '대상': model, '지표': k, 'FPR': fpr, '값': g[k].mean(),
                     '표준편차': g[k].std() if len(g) > 1 else np.nan})
for _, r in boot.iterrows():
    rows.append({'표': '부트스트랩', '구간': r['구간'], '대상': r['비교'], '지표': r['지표'],
                 '값': r['차이'], '하한': r['하한'], '상한': r['상한']})
for model, g in res_why.groupby('모델'):
    for k in ['찍은 거래', '기준 이상 비율', '기준 이상 중 찍힌 비율', '기준 미만에서 찍은 거래']:
        rows.append({'표': 'F1 원인', '구간': 'test', '대상': model, '지표': k, '값': g[k].mean(), '표준편차': g[k].std()})

summary = pd.DataFrame(rows, columns=['표', '구간', '대상', '지표', 'FPR', '값', '표준편차', '하한', '상한'])
out_path = f'{MID_DIR}/full_summary_B0_E4.csv'
summary.to_csv(out_path, index=False, encoding='utf-8-sig')
print('저장:', out_path, '|', len(summary), '행')
print(summary.groupby('표').size().to_string())

저장: /content/drive/MyDrive/26-2_AI_project/processed/mid/full_summary_B0_E4.csv | 116 행
표
F1       10
F1 원인     8
금액 규칙     8
부트스트랩    18
비교       30
성능       12
운영       30
